# Grover's Algorithm - Quantum Search
### Finding one marked item among 2^N with NVIDIA CUDA-Q
*Notebook 4 of 8 - CUDA-Q Quantum Algorithm Benchmarking Suite*

This notebook builds Grover's search algorithm in CUDA-Q, runs it on the backend you choose (CPU, NVIDIA GPU, multi-GPU or a quantum-hardware target), shows how the probability of finding the marked item grows and oscillates with the number of Grover iterations, and reports real timing and memory measurements.

**How to use this notebook:** run all cells from top to bottom (*Kernel -> Restart & Run All*).
Change parameters **only in Section 3**; every later cell reads them from there. The default
configuration runs immediately on a laptop CPU and much faster on an NVIDIA GPU.

| Section | Content |
|---|---|
| 1 | Problem statement and mathematical explanation |
| 2 | Import libraries (with installation instructions) |
| 3 | Input parameters - **the only cell you need to edit** |
| 4 | Example inputs |
| 5 | Benchmark utilities and backend initialization |
| 6 | Expected output (computed *before* running the quantum circuit) |
| 7 | Quantum circuit generation and visualization |
| 8 | Execution with live memory monitoring |
| 9 | Results and visualization |
| 10 | Final benchmark report |

---
## Section 1 - Problem statement and mathematical explanation

### What problem does the algorithm solve?
**Unstructured search.** We are given $M = 2^N$ items labelled by $N$-bit strings $x$ and a black box
(the *oracle*) that recognises the one item we want, $x^\ast$:

$$f(x) = \begin{cases} 1 & \text{if } x = x^\ast \\ 0 & \text{otherwise.} \end{cases}$$

The task: find $x^\ast$ with as few oracle calls as possible.

### Why is this problem important?
* Grover's algorithm gives a provable **quadratic speed-up** for every problem whose solutions are easy to
  *check* but hard to *find* (satisfiability, constraint problems, database-style search).
* Its core idea, **amplitude amplification**, is a building block of many other quantum algorithms
  (quantum counting, minimum finding, collision finding).
* It sets key-length margins in cryptography: a $k$-bit key needs about $2^{k/2}$ Grover iterations
  instead of $2^{k}$ classical guesses.

### What is the classical approach?
Without structure the only option is to test the items one by one: on average $M/2$ oracle calls,
$M$ in the worst case - the cost is $O(M)$.

### How does the quantum algorithm solve it?
1. **Uniform superposition.** Hadamard gates on all qubits create
   $|s\rangle = H^{\otimes N}|0\rangle^{\otimes N} = \frac{1}{\sqrt{M}}\sum_{x=0}^{M-1}|x\rangle$.
2. **Oracle** $O = I - 2|x^\ast\rangle\langle x^\ast|$ flips the sign of the marked amplitude,
   $O|x\rangle = (-1)^{f(x)}|x\rangle$. Circuit: $X$ on every qubit where $x^\ast$ has a 0, a
   multi-controlled $Z$, then the same $X$ gates again.
3. **Diffusion** $D = 2|s\rangle\langle s| - I$ ("inversion about the mean") turns the sign flip into a larger
   amplitude. Circuit: $H^{\otimes N}X^{\otimes N}\,(\text{multi-controlled }Z)\,X^{\otimes N}H^{\otimes N}$,
   which equals $D$ up to an irrelevant global phase.
4. Repeat the **Grover iteration** $G = D\,O$ exactly $k$ times and measure all qubits.

**Geometric picture.** Define $\sin\theta = 1/\sqrt{M}$. The state always lies in the plane spanned by
$|x^\ast\rangle$ and the uniform superposition of the unmarked items, and every iteration rotates it by
$2\theta$ towards $|x^\ast\rangle$:

$$G^k|s\rangle = \sin\big((2k+1)\theta\big)\,|x^\ast\rangle + \cos\big((2k+1)\theta\big)\,|\text{rest}\rangle,
\qquad P_k(x^\ast) = \sin^2\big((2k+1)\theta\big).$$

The success probability is largest after

$$k_{\text{opt}} = \operatorname{round}\!\left(\frac{\pi}{4\theta} - \frac{1}{2}\right) \approx \frac{\pi}{4}\sqrt{M}$$

iterations: $O(\sqrt{M})$ oracle calls instead of $O(M)$. **More** iterations rotate *past* the target and
the probability falls again (over-rotation); Section 9 shows this oscillation.

### Measurement
Measuring every qubit in the computational ($Z$) basis returns the bitstring $x$ with probability
$|\langle x|\psi\rangle|^2$ (Born rule). Bitstrings are printed with **qubit 0 first** (leftmost character).

### What is the expected solution?
For the default $N = 15$: $M = 32\,768$, $\theta \approx 0.005524$, $k_{\text{opt}} = 142$ and
$P(x^\ast) \approx 0.99999$ - essentially every shot returns the marked bitstring, whereas a classical search
needs 16 384 oracle calls on average.

---
## Section 2 - Import libraries

All quantum code uses **NVIDIA CUDA-Q** (`cudaq`). The other libraries are standard scientific Python.

**Installation** (Linux x86_64/aarch64 with or without an NVIDIA GPU, or macOS on Apple silicon for CPU-only simulation):

```bash
python3 -m venv cudaq-env && source cudaq-env/bin/activate     # Python >= 3.11
pip install --upgrade pip
pip install cudaq numpy scipy matplotlib psutil nvidia-ml-py jupyter
```

* `pip install cudaq` detects your CUDA driver and installs the matching CUDA-Q build (CUDA 12.x or 13.x).
  A GPU is **not** required: CPU simulators are always available.
* `nvidia-ml-py` provides NVML bindings used for GPU memory monitoring (optional on CPU-only machines).
* Multi-GPU / multi-node runs additionally need MPI; see the README of this repository.

The next cell imports everything, prints the versions found and warns about incompatible versions.

In [ ]:
# ---------------------------------------------------------------------------------
# Section 2 - Import libraries and check versions
# ---------------------------------------------------------------------------------
import os, sys, re, json, math, time, socket, platform      # standard library: files, timing, system info
import threading, subprocess, tempfile, datetime            # standard library: live memory monitor + report
import warnings                                             # used to hide two harmless CUDA-Q notices

# CUDA-Q 0.16 prints two informational warnings when imported: a "preview" notice for the
# bundled cudaq-logical package and a notice that sample/observe will change in a FUTURE
# release. Neither affects this notebook, so exactly these two messages are hidden.
warnings.filterwarnings("ignore", message=".*cudaq-logical is in preview.*")
warnings.filterwarnings("ignore", message=".*algorithmic primitives will change.*")

_missing = []                                               # collect missing packages for one clear message
try:
    import numpy as np                                      # arrays, linear algebra, random numbers
except ImportError:
    _missing.append("numpy")
try:
    import matplotlib                                       # plotting library
    import matplotlib.pyplot as plt                         # pyplot interface for figures
except ImportError:
    _missing.append("matplotlib")
try:
    import scipy                                            # scientific computing (optimizers, linear algebra)
    pass
except ImportError:
    _missing.append("scipy")
try:
    import psutil                                           # host RAM monitoring
except ImportError:
    _missing.append("psutil")
try:
    import cudaq                                            # NVIDIA CUDA-Q: kernels, simulators, QPU targets
except ImportError:
    _missing.append("cudaq")
if _missing:
    raise ImportError(f"Missing packages: {_missing}. Install them with:\n"
                      f"    pip install {' '.join(_missing)} nvidia-ml-py\n"
                      "CUDA-Q needs Linux (x86_64/aarch64) or macOS on Apple silicon (CPU only); see README.md.")

try:
    import pynvml                                           # NVIDIA Management Library bindings (GPU memory)
    HAVE_NVML = True
except ImportError:                                         # optional: only needed for GPU monitoring
    pynvml, HAVE_NVML = None, False

try:                                                        # live display inside Jupyter
    from IPython import get_ipython
    from IPython.display import display, Pretty
    _IN_JUPYTER = get_ipython() is not None and "IPKernelApp" in get_ipython().config
except Exception:                                           # running as a plain Python script
    display, Pretty, _IN_JUPYTER = None, None, False

# ---- Version report and compatibility check ----------------------------------------
_match = re.search(r"(\d+)\.(\d+)\.(\d+)", cudaq.__version__)
CUDAQ_VERSION = ".".join(_match.groups()) if _match else cudaq.__version__
_cudaq_tuple = tuple(int(x) for x in _match.groups()) if _match else (0, 0, 0)
print(f"Python      : {platform.python_version()} ({platform.system()} {platform.machine()})")
print(f"CUDA-Q      : {CUDAQ_VERSION}")
print(f"NumPy       : {np.__version__}")
print(f"SciPy       : {scipy.__version__}")
print(f"Matplotlib  : {matplotlib.__version__}")
print(f"psutil      : {psutil.__version__}")
print(f"nvidia-ml-py: {'installed' if HAVE_NVML else 'NOT installed (GPU memory monitoring disabled: pip install nvidia-ml-py)'}")
print(f"GPUs visible to CUDA-Q: {cudaq.num_available_gpus()}")
if sys.version_info < (3, 11):
    print("WARNING: CUDA-Q 0.16 wheels require Python >= 3.11.")
if _cudaq_tuple < (0, 12, 0):
    print("WARNING: this notebook was written and tested with CUDA-Q 0.16.0; versions older than 0.12 "
          "use a different operator API and are not supported. Upgrade with: pip install -U cudaq")
elif _cudaq_tuple != (0, 16, 0):
    print("Note: tested with CUDA-Q 0.16.0. Other versions should work; if an API error appears, "
          "compare with the README's 'Known limitations'.")

# ---------------------------------------------------------------------------------
# Plot style used by every figure: colour-blind-safe palette, hairline grid, one y-axis per chart
# ---------------------------------------------------------------------------------
# Categorical colours in a FIXED order (validated for colour-vision deficiency).
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BLUE, ORANGE, AQUA = SERIES_COLORS[0], SERIES_COLORS[1], SERIES_COLORS[2]
INK, INK_SECONDARY, INK_MUTED = "#0b0b0b", "#52514e", "#898781"   # text colours
GRID_COLOR, AXIS_COLOR, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"  # chart chrome
GOOD, CRITICAL = "#0ca30c", "#d03b3b"                                # reserved status colours
SEQUENTIAL_BLUE = matplotlib.colors.LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])          # magnitude
DIVERGING_BLUE_RED = matplotlib.colors.LinearSegmentedColormap.from_list(
    "div_blue_red", ["#184f95", "#6da7ec", "#f0efec", "#ec8a89", "#a32b2b"])      # signed values


def apply_plot_style():
    """Consistent, quiet chart style for every figure in this notebook."""
    plt.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": AXIS_COLOR, "axes.labelcolor": INK_SECONDARY, "axes.titlecolor": INK,
        "text.color": INK, "xtick.color": AXIS_COLOR, "ytick.color": AXIS_COLOR,
        "xtick.labelcolor": INK_SECONDARY, "ytick.labelcolor": INK_SECONDARY,
        "axes.grid": True, "grid.color": GRID_COLOR, "grid.linewidth": 0.8, "grid.linestyle": "-",
        "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
        "axes.prop_cycle": matplotlib.cycler(color=SERIES_COLORS),
        "lines.linewidth": 2.0, "lines.solid_capstyle": "round", "lines.markersize": 6,
        "legend.frameon": False, "font.size": 10, "axes.titlesize": 11, "figure.dpi": 100,
    })


apply_plot_style()

---
## Section 3 - Input parameters

This is the **only cell you need to edit**. Every later cell reads these variables.

In [ ]:
# =====================================================================================
# Section 3 - INPUT PARAMETERS  (edit only this cell, then re-run the notebook)
# =====================================================================================

# ---- Problem definition ---------------------------------------------------------------
N_QUBITS = 15                    # search space has 2**N_QUBITS items (15 -> 32,768); try 20, 25, 30 on GPUs
MARKED_BITSTRING = "example"     # "example" = reproducible example of Section 4, or your own item as an
                                 # N-character string of 0/1 (qubit 0 first), or an integer 0 <= x < 2**N

# ---- Algorithm parameters -------------------------------------------------------------
GROVER_ITERATIONS = "optimal"    # "optimal" = round(pi/(4*theta) - 1/2), or any integer k >= 0
RUN_ITERATION_SWEEP = True       # also measure P(marked) after 0, 1, 2, ... iterations (Section 9)
SWEEP_MAX_ITERATIONS = "auto"    # "auto" = 2*k_opt + 1 (one full rise and fall), or an integer
SWEEP_MAX_POINTS = 60            # at most this many points on the sweep curve

# ---- Backend and simulation method ---------------------------------------------------
BACKEND = "auto"                 # "auto" | "cpu" | "gpu" | "multi-gpu" | "hardware"
SIMULATION_METHOD = "statevector"   # this notebook supports: "statevector" | "mps" | "tensornet" | "density_matrix"
GPU_PRECISION = "fp64"            # "fp64" (16 B/amplitude) | "fp32" (8 B/amplitude, half the memory)
HARDWARE_PROVIDER = "quantinuum"  # only for BACKEND="hardware": "quantinuum", "ionq", "oqc", "anyon", "infleqtion", ...
HARDWARE_EMULATE = True           # True: local emulation (no account, no cost); False: real QPU (credentials needed)
HARDWARE_MACHINE = None           # provider device name, e.g. "H2-1E"; None = provider default

# ---- Shots and reproducibility --------------------------------------------------------
SHOTS = 2000                       # measurement repetitions; statistical error ~ 1/sqrt(SHOTS)
SEED = 42                         # fixes NumPy and CUDA-Q random numbers -> reproducible results

# ---- Memory monitoring ----------------------------------------------------------------
LIVE_MEMORY_DISPLAY = True        # live RAM / GPU memory line while the execution cell runs
MEMORY_SAMPLE_INTERVAL_S = 0.25   # sampling period in seconds (smaller = finer timeline)
MEMORY_SAFETY_FRACTION = 0.8      # refuse runs whose state needs more than 80% of free memory

# ---- Circuit drawing, export and report ---------------------------------------------------
DRAW_QUBITS = 10                  # qubits shown in the circuit drawing (maximum 10)
EXPORT_CIRCUIT = True             # save drawing + full circuit to EXPORT_DIR
EXPORT_FORMATS = ["txt", "png", "qir"]   # "txt", "png" (drawing), "qir", "qasm" (full circuit)
EXPORT_DIR = "exports/grover"     # output folder (created automatically)
SAVE_REPORT = True                # save the final benchmark report (JSON) and figure (PNG)

#### What every parameter means and what happens when you change it

| Parameter | Meaning | Effect of changing it |
|---|---|---|
| `N_QUBITS` | Number of qubits $N$; the search space has $M = 2^N$ items. | Memory doubles with every qubit (state vector: 16 x 2^N bytes in fp64). The number of iterations grows like $\sqrt{M} = 2^{N/2}$, so state-vector run time grows roughly like $2^{1.5N}$: N=15 takes seconds, N=25 minutes on a GPU, N >= 30 needs large or multiple GPUs. |
| `MARKED_BITSTRING` | The item to find (qubit 0 = leftmost character). | Grover's performance does **not** depend on which item is marked; any valid item gives the same success probability. A wrong length or characters other than 0/1 stop the notebook with an error. |
| `GROVER_ITERATIONS` | Number $k$ of oracle + diffusion repetitions. | `"optimal"` maximises $P(x^\ast)$. Fewer iterations lower the probability; more iterations over-rotate and the probability falls again (it is periodic in $k$). |
| `RUN_ITERATION_SWEEP` | Measure $P(x^\ast)$ as a function of $k$. | Costs about `SWEEP_MAX_ITERATIONS` extra iterations; switch off for very large N. |
| `SWEEP_MAX_ITERATIONS` / `SWEEP_MAX_POINTS` | Range and resolution of the sweep. | More points give a smoother curve. With the state-vector method the cost depends only on the range, not on the number of points. |
| `BACKEND` | Where the circuit runs: `"auto"` (NVIDIA GPU if CUDA-Q sees one, otherwise CPU), `"cpu"`, `"gpu"`, `"multi-gpu"`, `"hardware"`. | GPU state-vector simulation is typically 10-1000x faster than CPU for N >= 20 and is limited by GPU memory. `"multi-gpu"` pools the memory of several GPUs (launch with MPI, see README). `"hardware"` sends the circuit to a quantum computer provider (sampling only). |
| `SIMULATION_METHOD` | How the quantum state is stored (the allowed values for this notebook are listed in the input cell). `"statevector"`: all 2^N amplitudes, exact. `"mps"`: matrix product state, compressed (GPU). `"tensornet"`: exact tensor-network contraction (GPU). `"density_matrix"`: 4^N entries, supports noise (CPU). | Changes memory and run time: state vector needs 16 x 2^N bytes (fp64), density matrix 16 x 4^N bytes, MPS grows with entanglement instead of with N. An unsupported combination stops with an explanation. |
| `GPU_PRECISION` | `"fp64"` (16 bytes per amplitude) or `"fp32"` (8 bytes). GPU targets only; CPU targets always use fp64. | fp32 halves memory (one more qubit fits) and is faster, with ~1e-7 relative precision instead of ~1e-16. |
| `HARDWARE_PROVIDER` / `HARDWARE_EMULATE` / `HARDWARE_MACHINE` | Used only when `BACKEND="hardware"`: provider target (`"quantinuum"`, `"ionq"`, `"oqc"`, `"anyon"`, `"infleqtion"`, ...), local emulation switch, device name. | `HARDWARE_EMULATE=True` runs the provider's compilation flow locally with no account and no cost. `False` submits real jobs: needs credentials, may cost money, waits in a queue. |
| `SHOTS` | Number of times the circuit is executed and measured. | Statistical error of every measured probability shrinks like 1/sqrt(SHOTS). On simulators extra shots are cheap; on hardware cost grows linearly. |
| `SEED` | Random seed for NumPy and CUDA-Q's shot sampler. | Same seed gives identical results on re-run; change it to see statistical fluctuations. |
| `LIVE_MEMORY_DISPLAY` / `MEMORY_SAMPLE_INTERVAL_S` | Live RAM/GPU memory line during execution; sampling period in seconds. | Smaller intervals give a finer memory timeline; peaks shorter than the interval can be missed. |
| `MEMORY_SAFETY_FRACTION` | The run is refused if the quantum state would need more than this fraction of the free memory. | Raise it (e.g. 0.95) if you know the run fits; lower it to protect other work on the machine. |
| `DRAW_QUBITS` | Number of qubits used for the circuit **drawing** (at most 10). | Only changes the picture; execution always uses the full number of qubits. |
| `EXPORT_CIRCUIT` / `EXPORT_FORMATS` / `EXPORT_DIR` / `SAVE_REPORT` | Save the drawing (`"txt"`, `"png"`), the full circuit (`"qir"`, `"qasm"`), the report (JSON) and final figure (PNG) to `EXPORT_DIR`. | Set `EXPORT_CIRCUIT=False` / `SAVE_REPORT=False` to write nothing to disk. |

---
## Section 4 - Example inputs

**Default example (runs as is):** `N_QUBITS = 15`, `MARKED_BITSTRING = "example"`. The example item is a
pseudo-random 15-bit string generated from `SEED`, so it is reproducible but not "special" in any way. The
cell below prints it together with the derived quantities.

**How to change the example** - edit Section 3, for instance:

| Goal | Setting in Section 3 | What you will see |
|---|---|---|
| Your own item | `MARKED_BITSTRING = "110010101011100"` (exactly `N_QUBITS` characters) | that bitstring found with P ~ 1 |
| Integer input | `MARKED_BITSTRING = 12345` | binary of 12345 with N bits (qubit 0 = most significant bit) |
| Tiny instance | `N_QUBITS = 4`, `MARKED_BITSTRING = "1011"` | $k_{\text{opt}} = 3$, $P = 0.961$ - the whole circuit fits in the drawing |
| Over-rotation | `GROVER_ITERATIONS = 284` (= $2k_{\text{opt}}$ for N=15) | the marked item almost never appears |
| GPU benchmark | `N_QUBITS = 25`, `BACKEND = "gpu"` | $k_{\text{opt}} = 4\,549$ iterations on 33.5 million amplitudes |

In [ ]:
# ---------------------------------------------------------------------------------
# Section 4 - Build and validate the search problem from the Section 3 inputs
# ---------------------------------------------------------------------------------
if not isinstance(N_QUBITS, (int, np.integer)) or N_QUBITS < 2:
    raise ValueError("N_QUBITS must be an integer >= 2 (the multi-controlled Z needs at least 2 qubits).")
N_QUBITS = int(N_QUBITS)

if isinstance(MARKED_BITSTRING, str) and MARKED_BITSTRING.strip().lower() == "example":
    example_rng = np.random.default_rng(SEED)                       # reproducible example item
    marked_bitstring = "".join(str(b) for b in example_rng.integers(0, 2, N_QUBITS))
elif isinstance(MARKED_BITSTRING, (int, np.integer)) and not isinstance(MARKED_BITSTRING, bool):
    if not 0 <= int(MARKED_BITSTRING) < 2 ** N_QUBITS:
        raise ValueError(f"Integer MARKED_BITSTRING must satisfy 0 <= x < 2**{N_QUBITS} = {2 ** N_QUBITS}.")
    marked_bitstring = format(int(MARKED_BITSTRING), f"0{N_QUBITS}b")   # qubit 0 = most significant bit
else:
    marked_bitstring = str(MARKED_BITSTRING).strip()
    if len(marked_bitstring) != N_QUBITS or set(marked_bitstring) - {"0", "1"}:
        raise ValueError(f"MARKED_BITSTRING must contain exactly N_QUBITS={N_QUBITS} characters '0'/'1', "
                         f"got {marked_bitstring!r} (length {len(marked_bitstring)}).")
marked_bits = [int(c) for c in marked_bitstring]                    # list form passed to the CUDA-Q kernel

search_space = 2 ** N_QUBITS                                         # M = 2^N items
theta = math.asin(1.0 / math.sqrt(search_space))                     # rotation half-angle: sin(theta) = 1/sqrt(M)
k_optimal = max(0, round(math.pi / (4 * theta) - 0.5))               # iterations that maximise P(marked)
if isinstance(GROVER_ITERATIONS, str) and GROVER_ITERATIONS.strip().lower() == "optimal":
    iterations = k_optimal
elif isinstance(GROVER_ITERATIONS, (int, np.integer)) and int(GROVER_ITERATIONS) >= 0:
    iterations = int(GROVER_ITERATIONS)
else:
    raise ValueError("GROVER_ITERATIONS must be 'optimal' or a non-negative integer.")
if isinstance(SWEEP_MAX_ITERATIONS, str) and SWEEP_MAX_ITERATIONS.strip().lower() == "auto":
    sweep_max = 2 * k_optimal + 1                                    # one full rise and fall of P(marked)
else:
    sweep_max = int(SWEEP_MAX_ITERATIONS)
if not 0 < int(DRAW_QUBITS) <= 10:
    raise ValueError("DRAW_QUBITS must be between 1 and 10 (circuit drawings are limited to 10 qubits).")

print(f"Number of qubits N          : {N_QUBITS}")
print(f"Search space size M = 2^N   : {search_space:,}")
print(f"Marked bitstring x*         : {marked_bitstring}   (integer value {int(marked_bitstring, 2):,}, qubit 0 first)")
print(f"Rotation angle theta        : {theta:.6f} rad")
print(f"Optimal iterations k_opt    : {k_optimal:,}")
print(f"Iterations used             : {iterations:,}")
print(f"Classical expected queries  : {search_space / 2:,.0f} (average), {search_space:,} (worst case)")

---
## Section 5 - Benchmark utilities and backend initialization

The first cell below defines helper functions shared by all eight notebooks (you do not need to edit it):

* **Backend selection** - translates `BACKEND` + `SIMULATION_METHOD` into a real CUDA-Q target and
  refuses combinations CUDA-Q does not support (for example MPS on a CPU) with an explanation.
* **Memory estimation** - exact formulas for the memory needed by the quantum state.
* **Live memory monitor** - a small separate process samples host RAM (`psutil`) and the memory of
  every NVIDIA GPU (NVML) while the execution cell runs.
* **Timing** - separates circuit construction, compilation and execution using CUDA-Q's built-in tracer.
* **Circuit drawing / export** and the **final benchmark report**.

The second cell activates the selected backend and checks that the problem fits into memory.

| Your choice | CUDA-Q target used | Notes |
|---|---|---|
| `cpu` + `statevector` | `qpp-cpu` | OpenMP state vector, fp64 |
| `cpu` + `density_matrix` | `density-matrix-cpu` | mixed states and noise, fp64, 4^N memory |
| `cpu` + `stabilizer` | `stim` | Clifford circuits + Pauli noise only (noise notebook) |
| `gpu` + `statevector` | `nvidia` (option fp32/fp64) | cuStateVec, one GPU |
| `gpu` + `trajectory` | `nvidia` + noise model | Monte-Carlo noise trajectories (noise notebook) |
| `gpu` + `mps` | `tensornet-mps` | cuTensorNet matrix product state |
| `gpu` + `tensornet` | `tensornet` | cuTensorNet exact contraction |
| `multi-gpu` + `statevector` | `nvidia` (option `mgpu,fp64`) | state vector split over GPUs; needs `mpiexec` |
| `multi-gpu` + `tensornet` | `tensornet` | contraction distributed over GPUs; needs `mpiexec` |
| `hardware` | provider target (`quantinuum`, `ionq`, ...) | sampling only; `emulate=True` runs locally |

In [ ]:
# =====================================================================================
# BENCHMARK & MONITORING UTILITIES  (this cell is identical in all 8 notebooks)
# -------------------------------------------------------------------------------------
# You do not need to edit this cell. It defines helper functions used later:
#   resolve_backend()       -> turns BACKEND + SIMULATION_METHOD into a real CUDA-Q target
#   initialize_backend()    -> calls cudaq.set_target() safely
#   estimate_memory_bytes() -> memory needed to store the quantum state (exact formulas)
#   check_memory()          -> compares that requirement with free RAM / GPU memory
#   MemoryMonitor           -> live RAM + GPU memory monitoring during execution
#   profile_first_launch()  -> splits a kernel launch into construction/compilation/execution
#   measure_gate_time()     -> short calibration used for execution-time estimates
#   show_circuit()          -> prints a <=10-qubit circuit drawing and exports it
#   circuit_statistics()    -> gate counts/depth of the FULL circuit (no simulation needed)
#   show_report()           -> prints and saves the final benchmark report
# =====================================================================================

# ---------------------------------------------------------------------------------
# 1. Backend selection
# ---------------------------------------------------------------------------------
# (device, simulation method) -> (CUDA-Q target name, does the target accept fp32/fp64?)
# Only combinations that CUDA-Q actually ships are listed. Anything else raises a clear
# error instead of silently running on a different simulator.
_TARGET_TABLE = {
    ("cpu", "statevector"): ("qpp-cpu", False),             # OpenMP state vector (always fp64)
    ("cpu", "density_matrix"): ("density-matrix-cpu", False),  # exact mixed states (noise), fp64
    ("cpu", "stabilizer"): ("stim", False),                 # Clifford circuits + Pauli noise only
    ("gpu", "statevector"): ("nvidia", True),               # cuStateVec on one GPU
    ("gpu", "trajectory"): ("nvidia", True),                # cuStateVec + noise = Monte-Carlo trajectories
    ("gpu", "mps"): ("tensornet-mps", True),                # cuTensorNet matrix product state
    ("gpu", "tensornet"): ("tensornet", True),              # cuTensorNet exact tensor-network contraction
    ("multi-gpu", "statevector"): ("nvidia", True),         # one state vector split across GPUs (MPI)
    ("multi-gpu", "tensornet"): ("tensornet", True),        # contraction distributed across GPUs (MPI)
}

# Explanations for combinations CUDA-Q does not support (shown to the user on error).
_UNSUPPORTED_HINTS = {
    ("cpu", "mps"): "CUDA-Q's MPS simulator (target 'tensornet-mps') needs an NVIDIA GPU. "
                    "Use BACKEND='gpu', or SIMULATION_METHOD='statevector' on CPU.",
    ("cpu", "tensornet"): "CUDA-Q's tensor-network simulator (target 'tensornet') needs an NVIDIA GPU.",
    ("cpu", "trajectory"): "Trajectory-based noisy simulation runs on GPUs in CUDA-Q. On CPU use "
                           "'density_matrix' (exact, small N) or 'stabilizer' (Clifford circuits).",
    ("gpu", "density_matrix"): "CUDA-Q's density-matrix simulator is CPU-only ('density-matrix-cpu'). "
                               "On GPUs noisy circuits are simulated with 'trajectory'.",
    ("gpu", "stabilizer"): "The stabilizer simulator ('stim') is CPU-only. Use BACKEND='cpu'.",
    ("multi-gpu", "mps"): "The MPS simulator runs on a single GPU. Use BACKEND='gpu'.",
    ("multi-gpu", "density_matrix"): "The density-matrix simulator is CPU-only. Use BACKEND='cpu'.",
    ("multi-gpu", "stabilizer"): "The stabilizer simulator ('stim') is CPU-only. Use BACKEND='cpu'.",
    ("multi-gpu", "trajectory"): "Use BACKEND='gpu' for trajectory simulation in this notebook.",
}


def mpi_rank_info():
    """Return (rank, number_of_ranks). (0, 1) means a normal single-process run."""
    try:
        if cudaq.mpi.is_initialized():                       # CUDA-Q already started MPI
            return cudaq.mpi.rank(), cudaq.mpi.num_ranks()
    except Exception:                                        # cudaq.mpi may be unavailable
        pass
    # Common launcher environment variables (Open MPI, MPICH/Intel MPI, Slurm)
    for rank_var, size_var in (("OMPI_COMM_WORLD_RANK", "OMPI_COMM_WORLD_SIZE"),
                               ("PMI_RANK", "PMI_SIZE"), ("SLURM_PROCID", "SLURM_NTASKS")):
        if rank_var in os.environ:
            return int(os.environ[rank_var]), int(os.environ.get(size_var, "1"))
    return 0, 1


def resolve_backend(backend, method, precision, allowed_methods, hardware_provider=None,
                    hardware_emulate=True, hardware_machine=None):
    """Translate the user's BACKEND / SIMULATION_METHOD choice into a concrete CUDA-Q target.

    Nothing is changed in CUDA-Q here; we only validate and describe the choice.
    Raises ValueError / RuntimeError with an explanation for unsupported choices.
    """
    backend = str(backend).strip().lower()                   # tolerate "GPU", " cpu " ...
    requested = backend                                      # remember the user's original choice
    method = str(method).strip().lower()
    precision = str(precision).strip().lower()
    num_gpus = cudaq.num_available_gpus()                    # GPUs visible to CUDA-Q (0 on CPU-only machines)

    if backend == "auto":                                    # "auto": use a GPU if present, else the CPU
        backend = "gpu" if num_gpus > 0 else "cpu"
    if backend not in ("cpu", "gpu", "multi-gpu", "hardware"):
        raise ValueError(f"BACKEND must be 'auto', 'cpu', 'gpu', 'multi-gpu' or 'hardware', got {backend!r}")
    if precision not in ("fp32", "fp64"):
        raise ValueError(f"GPU_PRECISION must be 'fp32' or 'fp64', got {precision!r}")

    if backend == "hardware":                                # real QPU (or its local emulator)
        provider = str(hardware_provider).strip().lower()
        if not cudaq.has_target(provider):
            raise ValueError(f"CUDA-Q has no target named {provider!r}. "
                             f"Available targets: {sorted(t.name for t in cudaq.get_targets())}")
        kwargs = {}
        if hardware_emulate:
            kwargs["emulate"] = True                         # run locally, no credentials, no cost
        if hardware_machine:
            kwargs["machine"] = hardware_machine             # e.g. "H2-1E" for Quantinuum
        return {"backend_requested": requested, "device": "hardware", "method": "hardware",
                "target": provider, "kwargs": kwargs, "precision": "n/a", "num_gpus": num_gpus,
                "description": f"{provider} QPU " + ("(LOCAL EMULATION - no credentials, no cost)"
                                                     if hardware_emulate else "(REAL HARDWARE - needs credentials, may cost money)")}

    if method not in allowed_methods:                        # notebook-specific restriction
        raise ValueError(f"SIMULATION_METHOD={method!r} is not supported by this notebook. "
                         f"Choose one of {allowed_methods}.")
    if backend in ("gpu", "multi-gpu") and num_gpus == 0:    # never touch GPU targets without a GPU
        raise RuntimeError(f"BACKEND={backend!r} was requested but CUDA-Q sees no NVIDIA GPU "
                           "(cudaq.num_available_gpus() == 0). Check `nvidia-smi`, the driver and "
                           "the CUDA-Q CUDA variant (see README), or use BACKEND='cpu'.")
    key = (backend, method)
    if key not in _TARGET_TABLE:
        raise ValueError(_UNSUPPORTED_HINTS.get(key, f"Combination {key} is not supported by CUDA-Q."))
    target, takes_precision = _TARGET_TABLE[key]

    rank, size = mpi_rank_info()
    note = ""
    if backend == "multi-gpu" and method == "statevector":
        if size > 1:                                         # launched with mpiexec -> pool all GPUs
            option = f"mgpu,{precision}"
        else:                                                # plain Jupyter process: one rank = one GPU
            option = precision
            note = ("Multi-GPU state-vector pooling needs one MPI rank per GPU (see README: "
                    "`mpiexec -np <GPUs> python notebook.py`). This single process uses ONE GPU.")
    else:
        option = precision if takes_precision else ""
    kwargs = {"option": option} if option else {}
    return {"backend_requested": requested, "device": backend, "method": method, "target": target,
            "kwargs": kwargs, "precision": precision if takes_precision else "fp64",
            "num_gpus": num_gpus, "mpi_ranks": size,
            "description": f"{target}" + (f" (option={option})" if option else "") + (f" - {note}" if note else "")}


def initialize_backend(cfg, seed=None):
    """Activate the CUDA-Q target described by cfg (from resolve_backend) and seed its RNG."""
    if cfg["device"] == "multi-gpu" and cfg.get("mpi_ranks", 1) > 1 and not cudaq.mpi.is_initialized():
        cudaq.mpi.initialize()                               # required before using MPI-based targets
    cudaq.set_target(cfg["target"], **cfg["kwargs"])         # switch simulator / QPU
    if seed is not None:
        cudaq.set_random_seed(int(seed))                     # reproducible shot sampling
    return cudaq.get_target()


# ---------------------------------------------------------------------------------
# 2. Memory requirement estimates and availability checks
# ---------------------------------------------------------------------------------
def format_bytes(num_bytes):
    """Human-readable size. Works for astronomically large Python integers as well."""
    if num_bytes is None:
        return "n/a"
    if num_bytes >= 10**18:                                  # beyond exabytes: use powers of ten
        return f"10^{math.log10(num_bytes):.1f} bytes"
    for unit, scale in (("TB", 1e12), ("GB", 1e9), ("MB", 1e6), ("KB", 1e3)):
        if num_bytes >= scale:
            return f"{num_bytes / scale:.2f} {unit}"
    return f"{int(num_bytes)} B"


def estimate_memory_bytes(n_qubits, method, precision="fp64"):
    """Bytes needed to STORE the simulated quantum state (simulator workspace not included).

    statevector / trajectory : 2^N complex amplitudes (8 B each in fp32, 16 B in fp64)
    density_matrix           : 4^N complex entries (16 B each, fp64)
    mps                      : upper bound N * 2 * chi^2 amplitudes, chi = CUDAQ_MPS_MAX_BOND (default 64)
    stabilizer               : ~2N^2 bits (tiny)
    tensornet / hardware     : not predictable -> None
    """
    bytes_per_amplitude = 8 if precision == "fp32" else 16
    if method in ("statevector", "trajectory"):
        return (2 ** n_qubits) * bytes_per_amplitude
    if method == "density_matrix":
        return (4 ** n_qubits) * 16
    if method == "mps":
        chi = int(os.environ.get("CUDAQ_MPS_MAX_BOND", "64"))
        return n_qubits * 2 * chi * chi * bytes_per_amplitude
    if method == "stabilizer":
        return max(1, (2 * n_qubits * n_qubits) // 8)
    return None


def _nvml_gpu_memory():
    """List of (used_bytes, total_bytes, name) per GPU, or (None, reason) if NVML is unavailable."""
    if not HAVE_NVML:
        return None, "nvidia-ml-py (pynvml) is not installed"
    try:
        pynvml.nvmlInit()
        result = []
        for i in range(pynvml.nvmlDeviceGetCount()):
            handle = pynvml.nvmlDeviceGetHandleByIndex(i)
            info = pynvml.nvmlDeviceGetMemoryInfo(handle)
            name = pynvml.nvmlDeviceGetName(handle)
            result.append((info.used, info.total, name.decode() if isinstance(name, bytes) else name))
        return result, "NVML OK"
    except Exception as exc:                                 # no driver / no GPU
        return None, f"NVML unavailable ({type(exc).__name__}: {exc})"


def check_memory(required_bytes, cfg, safety_fraction=0.8, extra_copies=0):
    """Print required vs available memory and stop (MemoryError) if the run cannot fit.

    extra_copies: additional full copies of the state the notebook makes (e.g. get_state()
    returns a host copy of the state vector), added on top of the simulator's own copy.
    """
    if required_bytes is None:
        print("Memory requirement: cannot be predicted for this method (depends on circuit structure).")
        return None
    host_copy = extra_copies * required_bytes
    if cfg["device"] in ("gpu", "multi-gpu"):
        gpus, status = _nvml_gpu_memory()
        if gpus is None:
            print(f"GPU memory check skipped: {status}. Required (state only): {format_bytes(required_bytes)}")
            return None
        used_gpus = gpus if (cfg["device"] == "multi-gpu" and cfg.get("mpi_ranks", 1) > 1) else gpus[:1]
        available = sum(total - used for used, total, _ in used_gpus)
        where = f"free memory on {len(used_gpus)} GPU(s)"
    else:                                                    # CPU simulators and local emulators use host RAM
        available = psutil.virtual_memory().available
        where = "available host RAM"
        required_bytes = required_bytes + host_copy
        host_copy = 0
    print(f"Estimated memory for the quantum state : {format_bytes(required_bytes)}")
    if host_copy:
        print(f"Extra host RAM for state copies        : {format_bytes(host_copy)}")
    print(f"{where:<39}: {format_bytes(available)}  (safety limit {safety_fraction:.0%})")
    if required_bytes > safety_fraction * available:
        raise MemoryError(f"The state needs {format_bytes(required_bytes)} but only "
                          f"{format_bytes(available)} is {where.split(' on ')[0]} (limit {safety_fraction:.0%}). "
                          "Reduce the number of qubits, use fp32, an MPS/tensor-network method, more GPUs, "
                          "or raise MEMORY_SAFETY_FRACTION if you know the run fits.")
    print("Memory check: OK - the state fits within the safety limit.")
    return available


# ---------------------------------------------------------------------------------
# 3. Live memory monitor
# ---------------------------------------------------------------------------------
# Some CUDA-Q calls (for example cudaq.get_state) hold Python's global interpreter lock,
# which would freeze an ordinary Python thread. The sampler therefore runs as a small
# separate process that writes one JSON line per sample to a temporary file. A light
# thread in this notebook reads that file and refreshes the live display.
_SAMPLER_SCRIPT = r'''
import json, sys, time, psutil
pid, interval, path = int(sys.argv[1]), float(sys.argv[2]), sys.argv[3]
handles, pynvml = [], None
try:
    import pynvml
    pynvml.nvmlInit()
    handles = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in range(pynvml.nvmlDeviceGetCount())]
except Exception:
    handles = []
proc = psutil.Process(pid)
with open(path, "a", buffering=1) as out:
    while True:
        try:
            rss = proc.memory_info().rss
        except psutil.Error:
            break
        vm = psutil.virtual_memory()
        gpu = []
        for h in handles:
            try:
                m = pynvml.nvmlDeviceGetMemoryInfo(h)
                gpu.append([m.used, m.total])
            except Exception:
                gpu.append([None, None])
        out.write(json.dumps({"t": time.time(), "rss": rss, "sys_used": vm.total - vm.available,
                              "sys_total": vm.total, "gpu": gpu}) + "\n")
        time.sleep(interval)
'''


class MemoryMonitor:
    """Samples host RAM (psutil) and per-GPU memory (NVML) while code runs.

    Usage:
        monitor = MemoryMonitor(interval_s=0.25, live=True)
        monitor.start()
        ... run quantum code ...
        summary = monitor.stop()
    """

    def __init__(self, interval_s=0.25, live=True, refresh_s=1.0):
        self.interval_s = max(0.05, float(interval_s))     # sampling period of the sampler process
        self.live = live                                    # refresh a one-line live display?
        self.refresh_s = max(0.2, float(refresh_s))         # display refresh period
        self.host = socket.gethostname()                    # node name (matters for multi-node runs)
        self.rank, self.nranks = mpi_rank_info()
        self.samples = []
        self.gpu_names = []
        gpus, self.gpu_status = _nvml_gpu_memory()
        if gpus:
            self.gpu_names = [name for _, _, name in gpus]
        self._proc = psutil.Process(os.getpid())

    def _sample_here(self):
        """One sample taken inside this process (used for baseline, final and fallback)."""
        vm = psutil.virtual_memory()
        gpus, _ = _nvml_gpu_memory()
        return {"t": time.time(), "rss": self._proc.memory_info().rss,
                "sys_used": vm.total - vm.available, "sys_total": vm.total,
                "gpu": [[u, t] for u, t, _ in gpus] if gpus else []}

    def _read_file_samples(self):
        try:
            with open(self._path) as fh:
                lines = fh.readlines()
        except OSError:
            return []
        parsed = []
        for line in lines:
            try:
                parsed.append(json.loads(line))
            except ValueError:                                # partially written last line
                pass
        return parsed

    def _format(self, sample, peak_rss, peak_sys):
        gb = 1e9
        text = (f"[{self.host} rank {self.rank}] t={sample['t'] - self._t0:6.1f}s | "
                f"RAM used {sample['sys_used'] / gb:6.2f}/{sample['sys_total'] / gb:.1f} GB "
                f"({100 * sample['sys_used'] / sample['sys_total']:4.1f}%, peak {peak_sys / gb:.2f}) | "
                f"process RSS {sample['rss'] / gb:5.2f} GB (peak {peak_rss / gb:.2f})")
        for i, (used, total) in enumerate(sample.get("gpu", [])):
            if used is not None:
                text += f" | GPU{i} {used / gb:5.2f}/{total / gb:.0f} GB ({100 * used / total:4.1f}%)"
        return text

    def _display_loop(self):
        while not self._stop_event.wait(self.refresh_s):
            samples = self._read_file_samples() if self._path else []
            if not samples:
                continue
            peak_rss = max(s["rss"] for s in samples)
            peak_sys = max(s["sys_used"] for s in samples)
            self._show(self._format(samples[-1], peak_rss, peak_sys))

    def _show(self, text):
        if self._handle is not None:
            self._handle.update(Pretty(text))               # update the same output line in Jupyter
        else:
            print(text, flush=True)                          # plain scripts: print a new line

    def start(self):
        self._t0 = time.time()
        self._stop_event = threading.Event()
        self._baseline = self._sample_here()
        fd, self._path = tempfile.mkstemp(prefix="cudaq_memmon_", suffix=".jsonl")
        os.close(fd)
        try:                                                 # preferred: independent sampler process
            self._sampler = subprocess.Popen(
                [sys.executable, "-c", _SAMPLER_SCRIPT, str(os.getpid()), str(self.interval_s), self._path],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            self.mode = "separate sampler process"
            deadline = time.time() + 5.0                     # wait for the first sample (<= 5 s)
            while time.time() < deadline and not self._read_file_samples():
                time.sleep(0.02)
        except Exception as exc:                              # fallback: in-process thread
            self._sampler = None
            self.mode = f"in-process thread (sampler process failed: {exc})"
            self._thread_sampler = threading.Thread(target=self._fallback_loop, daemon=True)
            self._thread_sampler.start()
        self._handle = None
        if self.live:
            if _IN_JUPYTER:
                self._handle = display(Pretty("starting live memory monitor ..."), display_id=True)
            self._display_thread = threading.Thread(target=self._display_loop, daemon=True)
            self._display_thread.start()
        return self

    def _fallback_loop(self):
        with open(self._path, "a", buffering=1) as out:
            while not self._stop_event.wait(self.interval_s):
                out.write(json.dumps(self._sample_here()) + "\n")

    def stop(self):
        final = self._sample_here()
        self._stop_event.set()
        if self._sampler is not None:
            self._sampler.terminate()
            try:
                self._sampler.wait(timeout=5)
            except subprocess.TimeoutExpired:
                self._sampler.kill()
        if self.live:
            self._display_thread.join(timeout=5)
        self.samples = [self._baseline] + self._read_file_samples() + [final]
        self.samples.sort(key=lambda s: s["t"])
        try:
            os.remove(self._path)
        except OSError:
            pass
        summary = self.summary()
        if self.live:
            self._show(self._format(final, summary["rss_peak_gb"] * 1e9, summary["ram_used_peak_gb"] * 1e9)
                       + "  <- monitoring finished")
        return summary

    def summary(self):
        """Baseline / peak values in GB and % for RAM, process RSS and every GPU."""
        gb = 1e9
        base = self.samples[0]
        result = {
            "node": self.host, "mpi_rank": self.rank, "mpi_ranks": self.nranks,
            "sampler": self.mode, "num_samples": len(self.samples),
            "duration_s": self.samples[-1]["t"] - self.samples[0]["t"],
            "ram_total_gb": base["sys_total"] / gb,
            "ram_used_baseline_gb": base["sys_used"] / gb,
            "ram_used_peak_gb": max(s["sys_used"] for s in self.samples) / gb,
            "rss_baseline_gb": base["rss"] / gb,
            "rss_peak_gb": max(s["rss"] for s in self.samples) / gb,
            "gpu_status": self.gpu_status, "gpus": [],
        }
        result["ram_used_peak_pct"] = 100 * result["ram_used_peak_gb"] / result["ram_total_gb"]
        num_gpus = max((len(s.get("gpu", [])) for s in self.samples), default=0)
        for i in range(num_gpus):
            used = [s["gpu"][i][0] for s in self.samples if len(s.get("gpu", [])) > i and s["gpu"][i][0] is not None]
            totals = [s["gpu"][i][1] for s in self.samples if len(s.get("gpu", [])) > i and s["gpu"][i][1]]
            if not used or not totals:
                continue
            result["gpus"].append({
                "index": i, "name": self.gpu_names[i] if i < len(self.gpu_names) else f"GPU{i}",
                "total_gb": totals[0] / gb, "used_baseline_gb": used[0] / gb, "used_peak_gb": max(used) / gb,
                "used_peak_pct": 100 * max(used) / totals[0]})
        return result

    def plot(self, ax=None, title="Memory during execution"):
        """Plot RAM, process RSS and per-GPU memory versus time."""
        if ax is None:
            _, ax = plt.subplots(figsize=(9, 3.2))
        t = np.array([s["t"] for s in self.samples]) - self.samples[0]["t"]
        ax.plot(t, [s["sys_used"] / 1e9 for s in self.samples], label="host RAM used (all processes)")
        ax.plot(t, [s["rss"] / 1e9 for s in self.samples], label="this notebook process (RSS)")
        num_gpus = max((len(s.get("gpu", [])) for s in self.samples), default=0)
        for i in range(num_gpus):
            pts = [(tt, s["gpu"][i][0] / 1e9) for tt, s in zip(t, self.samples)
                   if len(s.get("gpu", [])) > i and s["gpu"][i][0] is not None]
            if pts:
                ax.plot(*zip(*pts), label=f"GPU{i} memory used")
        ax.set_ylim(bottom=0)                                 # start at zero: do not exaggerate small changes
        ax.set_xlabel("time since start of execution [s]")
        ax.set_ylabel("memory [GB]")
        ax.set_title(title)
        ax.legend(fontsize=8, loc="best")
        ax.grid(alpha=0.3)
        return ax


def print_memory_summary(mem):
    """Print the memory section of the benchmark (per node and per GPU)."""
    print(f"Node {mem['node']} (MPI rank {mem['mpi_rank']}/{mem['mpi_ranks']}), sampler: {mem['sampler']}, "
          f"{mem['num_samples']} samples over {mem['duration_s']:.1f} s")
    print(f"  Host RAM used : baseline {mem['ram_used_baseline_gb']:.2f} GB -> peak {mem['ram_used_peak_gb']:.2f} GB "
          f"of {mem['ram_total_gb']:.1f} GB ({mem['ram_used_peak_pct']:.1f}%)")
    print(f"  Process RSS   : baseline {mem['rss_baseline_gb']:.2f} GB -> peak {mem['rss_peak_gb']:.2f} GB "
          f"(increase {mem['rss_peak_gb'] - mem['rss_baseline_gb']:+.2f} GB)")
    if mem["gpus"]:
        for g in mem["gpus"]:
            print(f"  GPU{g['index']} ({g['name']}): baseline {g['used_baseline_gb']:.2f} GB -> peak "
                  f"{g['used_peak_gb']:.2f} GB of {g['total_gb']:.1f} GB ({g['used_peak_pct']:.1f}%)")
    else:
        print(f"  GPU memory    : not monitored - {mem['gpu_status']}")
    if mem["mpi_ranks"] > 1:
        print("  (multi-node/multi-rank run: every MPI rank prints its own node's line)")


# ---------------------------------------------------------------------------------
# 4. Timing: construction vs compilation vs execution
# ---------------------------------------------------------------------------------
# CUDA-Q (>= 0.16) contains a span tracer that records how long each internal phase
# takes. We use it for the FIRST launch of a kernel only:
#   construction = Python source -> Quake IR      (span "PyKernelDecorator.compile")
#   compilation  = MLIR passes + LLVM JIT          (span "Compiler::runPassPipeline")
#   execution    = running the simulation/QPU job  (spans "streamlinedLaunchModule" and
#                  "...finalizeExecutionContext", where get_state() flushes queued gates)
# The tracer adds ~1 microsecond per gate, so later launches are timed without it.
_EXECUTION_SPANS = {"streamlinedLaunchModule", "DefaultPlatform::finalizeExecutionContext",
                    "BasicExecutionManager::finalizeExecutionContext"}


def _span_union_seconds(events, names):
    """Total time covered by spans with the given names (nested/overlapping spans counted once)."""
    intervals = sorted((e["ts"], e["ts"] + e["dur"]) for e in events if e.get("name") in names and "dur" in e)
    total, start, end = 0.0, None, None
    for s, e in intervals:
        if end is None or s > end:                            # a new, separate interval begins
            if end is not None:
                total += end - start
            start, end = s, e
        else:                                                 # overlapping/nested: extend
            end = max(end, e)
    if end is not None:
        total += end - start
    return total / 1e6                                        # tracer timestamps are microseconds


def profile_first_launch(launch):
    """Run launch() once under the CUDA-Q tracer. Returns (result, phase-times dict)."""
    tracer, backend_obj = None, None
    try:
        from cudaq.util import trace as tracer
        backend_obj = tracer.ChromeBackend()                  # in-memory event recorder
        tracer.set_backend(backend_obj)
    except Exception:                                         # older CUDA-Q: no tracer
        tracer = None
    t_start = time.perf_counter()
    try:
        result = launch()
    finally:
        wall = time.perf_counter() - t_start
        if tracer is not None:
            tracer.reset_backend()                            # stop recording
    phases = {"wall_s": wall, "construction_s": None, "compilation_s": None, "execution_s": None,
              "method": "wall clock only (tracer not available)"}
    if backend_obj is not None:
        events = json.loads(backend_obj.to_json()).get("traceEvents", [])
        construct = _span_union_seconds(events, {"cudaq.kernel.kernel_decorator.PyKernelDecorator.compile"})
        compile_ = _span_union_seconds(events, {"Compiler::runPassPipeline"})
        execute = _span_union_seconds(events, _EXECUTION_SPANS)
        if compile_ > 0 or execute > 0:
            phases.update(construction_s=construct, compilation_s=compile_, execution_s=execute,
                          method="CUDA-Q span tracer (first launch)")
        else:
            phases["method"] = "wall clock only (this target does not report compile/execute spans)"
    return result, phases


def fmt_seconds(value):
    return "not measured" if value is None else f"{value:.4f} s"


# ---------------------------------------------------------------------------------
# 5. Execution-time estimate: measure the cost of one gate on the active backend
# ---------------------------------------------------------------------------------
@cudaq.kernel
def _calibration_kernel(n: int, repetitions: int):
    qubits = cudaq.qvector(n)
    for _ in range(repetitions):
        for i in range(n):
            ry(0.1, qubits[i])
    mz(qubits)


def measure_gate_time(cfg, n_qubits):
    """Seconds per gate at n_qubits, measured on the active target (rough, +-3x).

    Two short runs with different gate counts are timed; their difference removes fixed
    overheads. If n_qubits is large, the calibration runs on fewer qubits and is scaled by
    2^dN (state vector) or 4^dN (density matrix) because cost grows with the state size.
    """
    method = cfg["method"]
    if cfg["device"] == "hardware":
        return None, "not estimated: hardware time is dominated by queueing and provider limits"
    if method in ("mps", "tensornet", "stabilizer"):
        return None, f"not estimated: '{method}' cost depends on entanglement/circuit structure"
    cap = 9 if method == "density_matrix" else (14 if cfg["device"] == "cpu" else 24)
    n_cal = max(2, min(n_qubits, cap))
    def timed(repetitions):
        start = time.perf_counter()
        cudaq.sample(_calibration_kernel, n_cal, repetitions, shots_count=1)
        return time.perf_counter() - start
    timed(1)                                                  # warm-up: JIT compilation
    r1, r2 = 2, 12
    t1 = min(timed(r1) for _ in range(2))
    t2 = min(timed(r2) for _ in range(2))
    per_gate = max((t2 - t1) / ((r2 - r1) * n_cal), 1e-9)
    growth = 4 if method == "density_matrix" else 2
    scaled = per_gate * growth ** (n_qubits - n_cal)
    detail = f"measured {per_gate * 1e3:.4f} ms/gate at {n_cal} qubits"
    if n_qubits > n_cal:
        detail += f", extrapolated x{growth}^{n_qubits - n_cal} to {n_qubits} qubits"
    return scaled, detail


def format_duration(seconds):
    if seconds is None:
        return "not estimated"
    if seconds < 1:
        return f"{seconds * 1e3:.0f} ms"
    if seconds < 120:
        return f"{seconds:.1f} s"
    if seconds < 7200:
        return f"{seconds / 60:.1f} min"
    if seconds < 3 * 86400:
        return f"{seconds / 3600:.1f} h"
    return f"{seconds / 86400:.1e} days"


# ---------------------------------------------------------------------------------
# 6. Circuit drawing, statistics and export
# ---------------------------------------------------------------------------------
def circuit_statistics(kernel, *args):
    """Gate counts and depth of the FULL circuit via cudaq.estimate_resources (no simulation)."""
    try:
        res = cudaq.estimate_resources(kernel, *args)
        by_type = dict(res.to_dict())
        measurements = sum(v for k, v in by_type.items() if k in ("mz", "mx", "my"))
        return {"qubits": res.num_qubits, "gates (excluding measurements)": res.count() - measurements,
                "measurements": measurements, "depth": res.depth,
                "multi-qubit gates": res.multi_qubit_gate_count, "gate counts by type": by_type}
    except Exception as exc:
        return {"error": f"estimate_resources failed: {type(exc).__name__}: {exc}"}


def _render_text_png(text, path):
    lines = text.splitlines() or [""]
    width, height = max(len(line) for line in lines), len(lines)
    fig_w, fig_h = max(4.0, 0.075 * width), max(1.0, 0.19 * height)
    dpi = int(min(150, 30000 / (fig_w * 1.0)))               # keep huge drawings below ~30k pixels
    fig = plt.figure(figsize=(fig_w, fig_h))
    fig.text(0.0, 1.0, text, family="DejaVu Sans Mono", fontsize=9, va="top", ha="left")
    fig.savefig(path, dpi=max(dpi, 30), bbox_inches="tight", facecolor="white")
    plt.close(fig)


def show_circuit(kernel, args, n_shown, n_full, title, export=False, formats=("txt", "png", "qir"),
                 out_dir="exports", basename="circuit", full_kernel=None, full_args=()):
    """Print the circuit drawing of `kernel(*args)` (built with n_shown <= 10 qubits) and export it.

    Export formats:
      "txt"  - the drawing as UTF-8 text          "png"  - the drawing as an image
      "qir"  - the FULL N-qubit circuit (full_kernel(*full_args)) as QIR, the LLVM-based
               intermediate representation CUDA-Q sends to simulators and QPUs
      "qasm" - the FULL circuit as OpenQASM 2.0 (only for circuits CUDA-Q can unroll into
               plain gates; otherwise skipped with an explanation)
    """
    drawing = cudaq.draw(kernel, *args)
    if n_full > n_shown:
        banner = (f"CIRCUIT VISUALIZATION EXAMPLE: {n_shown}-qubit version of the circuit. "
                  f"This is NOT the complete {n_full}-qubit circuit that will be executed.")
    else:
        banner = f"Circuit drawing ({n_shown} qubits). {title}"
    print("=" * min(len(banner), 110))
    print(banner)
    print("=" * min(len(banner), 110))
    print(drawing)
    saved = []
    if export:
        os.makedirs(out_dir, exist_ok=True)
        stem = os.path.join(out_dir, f"{basename}_{n_shown}q")
        if "txt" in formats:
            with open(stem + ".txt", "w", encoding="utf-8") as fh:
                fh.write(banner + "\n\n" + drawing)
            saved.append(stem + ".txt")
        if "png" in formats:
            try:
                _render_text_png(banner + "\n\n" + drawing, stem + ".png")
                saved.append(stem + ".png")
            except Exception as exc:
                print(f"PNG export skipped: {exc}")
        if "qir" in formats and full_kernel is not None:
            try:
                qir = cudaq.translate(full_kernel, *full_args, format="qir")
                if "translation failed" in qir:
                    raise RuntimeError("CUDA-Q could not translate this circuit to QIR")
                path = os.path.join(out_dir, f"{basename}_{n_full}q_full.ll")
                with open(path, "w") as fh:
                    fh.write(qir)
                saved.append(path)
            except Exception as exc:
                print(f"QIR export skipped: {type(exc).__name__}: {exc}")
        if "qasm" in formats and full_kernel is not None:
            try:
                specialised = cudaq.synthesize(full_kernel, *full_args) if full_args else full_kernel
                qasm = cudaq.translate(specialised, format="openqasm2")
                if "translation failed" in qasm:
                    raise RuntimeError("translation failed")
                path = os.path.join(out_dir, f"{basename}_{n_full}q_full.qasm")
                with open(path, "w") as fh:
                    fh.write(qasm)
                saved.append(path)
            except Exception as exc:
                print("OpenQASM 2.0 export skipped: CUDA-Q 0.16 can only emit OpenQASM for circuits that "
                      f"unroll into plain gates without sub-kernels or list arguments ({type(exc).__name__}). "
                      "Use the 'qir' export instead.")
        for path in saved:
            print(f"Exported: {path}")
    return drawing


# ---------------------------------------------------------------------------------
# 7. Final benchmark report
# ---------------------------------------------------------------------------------
def build_report(algorithm, cfg, n_qubits, shots, phases, mem, extra_times=None):
    """Collect the standard benchmark fields into an ordered dict."""
    target = cudaq.get_target()
    report = {
        "Algorithm": algorithm,
        "Date (UTC)": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
        "CUDA-Q version": CUDAQ_VERSION,
        "Number of qubits": n_qubits,
        "Backend (requested -> used)": f"{cfg['backend_requested']} -> {cfg['device']}",
        "CUDA-Q target": target.name + (f" [{cfg['kwargs']}]" if cfg["kwargs"] else ""),
        "Simulation method": cfg["method"],
        "Numerical precision": cfg["precision"],
        "Number of shots": shots,
        "Circuit construction time": fmt_seconds(phases.get("construction_s")),
        "Compilation time (MLIR + JIT)": fmt_seconds(phases.get("compilation_s")),
        "Quantum execution time": fmt_seconds(phases.get("execution_s")),
    }
    for key, value in (extra_times or {}).items():
        report[key] = fmt_seconds(value)
    report["Total wall time of execution cell"] = fmt_seconds(phases.get("total_wall_s"))
    report["Timing method"] = phases.get("method", "")
    report["Peak host RAM used"] = f"{mem['ram_used_peak_gb']:.2f} GB ({mem['ram_used_peak_pct']:.1f}% of {mem['ram_total_gb']:.1f} GB)"
    report["Peak notebook process RSS"] = f"{mem['rss_peak_gb']:.2f} GB (baseline {mem['rss_baseline_gb']:.2f} GB)"
    if mem["gpus"]:
        for g in mem["gpus"]:
            report[f"Peak GPU{g['index']} memory ({g['name']})"] = (f"{g['used_peak_gb']:.2f} GB "
                                                                   f"({g['used_peak_pct']:.1f}% of {g['total_gb']:.1f} GB)")
    else:
        report["Peak GPU memory"] = f"not available - {mem['gpu_status']}"
    report["Node / MPI rank"] = f"{mem['node']} / rank {mem['mpi_rank']} of {mem['mpi_ranks']}"
    return report


def show_report(report, save=True, out_dir="exports", name="benchmark_report", figure=None):
    """Pretty-print the report and optionally save it as JSON (+ figure as PNG)."""
    width = max(len(str(k)) for k in report)
    print("=" * 100)
    print(f"{'FINAL BENCHMARK REPORT':^100}")
    print("=" * 100)
    for key, value in report.items():
        print(f"{str(key):<{width}} : {value}")
    print("=" * 100)
    if save:
        os.makedirs(out_dir, exist_ok=True)
        json_path = os.path.join(out_dir, f"{name}.json")
        with open(json_path, "w") as fh:
            json.dump({str(k): (v if isinstance(v, (int, float, str, bool, type(None))) else str(v))
                       for k, v in report.items()}, fh, indent=2)
        print(f"Report saved to {json_path}")
        if figure is not None:
            fig_path = os.path.join(out_dir, f"{name}.png")
            figure.savefig(fig_path, dpi=120, bbox_inches="tight")
            print(f"Final figure saved to {fig_path}")


def bit_reverse_order(vector, n_qubits):
    """Convert between CUDA-Q's state-vector order and 'qubit 0 = most significant bit'.

    cudaq.get_state() stores amplitude index i = sum_q b_q * 2^q (qubit 0 is the LEAST
    significant bit), while measured bitstrings are printed with qubit 0 FIRST (leftmost).
    Reversing the qubit axes converts one convention into the other (and back).
    """
    vector = np.asarray(vector)
    return vector.reshape([2] * n_qubits).transpose(list(range(n_qubits))[::-1]).reshape(-1)


print("Benchmark utilities loaded.")

In [ ]:
# ---------------------------------------------------------------------------------
# Activate the selected backend and print what is available on this machine
# ---------------------------------------------------------------------------------
ALLOWED_METHODS = ["statevector", "mps", "tensornet", "density_matrix"]   # simulation methods that make sense for this notebook
np.random.seed(SEED)                          # NumPy random numbers are reproducible too

backend_cfg = resolve_backend(BACKEND, SIMULATION_METHOD, GPU_PRECISION, ALLOWED_METHODS,
                              HARDWARE_PROVIDER, HARDWARE_EMULATE, HARDWARE_MACHINE)
active_target = initialize_backend(backend_cfg, seed=SEED)   # calls cudaq.set_target(...)

print(f"Requested backend : {BACKEND}  ->  device: {backend_cfg['device']}")
print(f"CUDA-Q target     : {active_target.name}  ({backend_cfg['description']})")
print(f"Simulation method : {backend_cfg['method']},  precision: {backend_cfg['precision']}")
print(f"GPUs visible      : {backend_cfg['num_gpus']},  CPU cores: {os.cpu_count()},  "
      f"host RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")
gpu_list, gpu_status = _nvml_gpu_memory()
for i, (used, total, name) in enumerate(gpu_list or []):
    print(f"  GPU{i}: {name}, {total / 1e9:.1f} GB total, {used / 1e9:.2f} GB in use")
if not gpu_list:
    print(f"  GPU monitoring: {gpu_status}")

---
## Section 6 - Expected output (before execution)

**Expected graphs (Section 9)**
1. *Measured bitstrings* - one dominant bar at the marked bitstring holding almost all `SHOTS` counts.
2. *P(marked) versus the number of iterations* - a $\sin^2((2k+1)\theta)$ curve: starts at $1/M$, peaks near
   $k_{\text{opt}}$ and returns towards zero at about $2k_{\text{opt}}$. The simulated points must lie on the theory curve.
3. *Memory timeline* of the execution.

**Expected numbers** follow exactly from the formulas of Section 1 and are computed below: the success
probability, the expected number of marked counts and their binomial spread $\sqrt{S\,P(1-P)}$ for $S$ shots.

**Execution time and memory are not guessed.** The memory of the quantum state follows from an exact formula;
the execution time is *estimated* from a short calibration on your backend (time per gate) multiplied by the
number of gates. Treat the time as a rough estimate (factor ~3), because gate types, caching and Python
overhead differ from the calibration circuit.

In [ ]:
# ---------------------------------------------------------------------------------
# Section 6 - Expected results, memory requirement and execution-time estimate
# ---------------------------------------------------------------------------------
p_theory = math.sin((2 * iterations + 1) * theta) ** 2               # P(marked) after `iterations` iterations
expected_counts = SHOTS * p_theory                                   # expected number of shots on x*
count_spread = math.sqrt(SHOTS * p_theory * (1 - p_theory))          # binomial standard deviation
print("Expected result")
print(f"  P(marked) after {iterations} iterations : {p_theory:.8f}")
print(f"  Expected marked counts            : {expected_counts:.1f} +- {count_spread:.1f} of {SHOTS} shots")
print(f"  Expected most frequent bitstring  : {marked_bitstring}" if p_theory > 0.5 else
      "  P(marked) < 0.5: the marked item is NOT expected to dominate (check GROVER_ITERATIONS)")

# ---- Memory: the state vector (or MPS/density matrix) of N qubits ----------------------------
uses_state_sweep = RUN_ITERATION_SWEEP and backend_cfg["method"] == "statevector" and backend_cfg["device"] != "hardware"
state_bytes = estimate_memory_bytes(N_QUBITS, backend_cfg["method"], backend_cfg["precision"])
print("\nExpected memory")
check_memory(state_bytes, backend_cfg, MEMORY_SAFETY_FRACTION,
             extra_copies=1 if uses_state_sweep else 0)              # the sweep holds two states at once

# ---- Execution-time estimate: gates x measured time per gate --------------------------------
zeros = marked_bitstring.count("0")
gates_per_iteration = 4 * N_QUBITS + 2 * zeros + 2                    # H and X layers, oracle X gates, 2 multi-controlled Z
main_gates = N_QUBITS + iterations * gates_per_iteration              # initial H layer + iterations
sweep_gates = (sweep_max * gates_per_iteration) if RUN_ITERATION_SWEEP else 0
if RUN_ITERATION_SWEEP and not uses_state_sweep:                      # sampled sweep re-runs circuits from scratch
    sweep_gates = int(np.sum(np.linspace(0, sweep_max, num=min(SWEEP_MAX_POINTS, 12)))) * gates_per_iteration
seconds_per_gate, timing_note = measure_gate_time(backend_cfg, N_QUBITS)
print("\nExpected execution time (ESTIMATE)")
print(f"  Gates in the main circuit : {main_gates:,}   (per iteration: {gates_per_iteration})")
print(f"  Gates in the sweep        : {sweep_gates:,}")
print(f"  Calibration               : {timing_note}")
if seconds_per_gate is not None:
    print(f"  Main Grover run           : ~{format_duration(main_gates * seconds_per_gate)}")
    print(f"  Iteration sweep           : ~{format_duration(sweep_gates * seconds_per_gate)}")

---
## Section 7 - Quantum circuit generation

The CUDA-Q kernels below implement Section 1 line by line. They are created inside a function
(`build_grover_kernels`) so that the execution cell can build **fresh** kernels and measure the circuit
construction and compilation time honestly (CUDA-Q caches compiled kernels).

* `grover_iteration` - oracle $O$ followed by diffusion $D$ on a register of qubits.
* `grover_search` - $H^{\otimes N}$, then $k$ iterations, then measurement of all qubits.
* `uniform_superposition` / `continue_iterations` - used by the iteration sweep: the state after $k$
  iterations is handed to the next kernel (`cudaq.qvector(state)`), so the whole sweep costs only
  `SWEEP_MAX_ITERATIONS` iterations instead of re-running every circuit from the start.

In [ ]:
# ---------------------------------------------------------------------------------
# Section 7a - CUDA-Q kernels for Grover's algorithm
# ---------------------------------------------------------------------------------
def build_grover_kernels():
    """Create and return fresh CUDA-Q kernels (called again in the execution cell)."""

    @cudaq.kernel
    def grover_iteration(qubits: cudaq.qview, marked: list[int]):
        """One Grover iteration G = D * O on `qubits`; `marked` lists the bits of x* (qubit 0 first)."""
        n = qubits.size()                            # number of qubits in the register
        # ---- Oracle O: multiply the amplitude of |x*> by -1 -------------------------------------
        for i in range(n):
            if marked[i] == 0:
                x(qubits[i])                         # X where x* has a 0: maps |x*> onto |11...1>
        z.ctrl(qubits[0:n - 1], qubits[n - 1])       # Z on the last qubit if all others are 1: flips the sign of |11...1>
        for i in range(n):
            if marked[i] == 0:
                x(qubits[i])                         # undo the X gates: the sign flip now sits on |x*>
        # ---- Diffusion D = 2|s><s| - I (inversion about the mean) ---------------------------------
        h(qubits)                                    # |s> -> |00...0>
        x(qubits)                                    # |00...0> -> |11...1>
        z.ctrl(qubits[0:n - 1], qubits[n - 1])       # sign flip of |11...1>, i.e. of |s> in the original basis
        x(qubits)                                    # undo X
        h(qubits)                                    # undo H (result: -D, the minus sign is a global phase)

    @cudaq.kernel
    def grover_search(n: int, marked: list[int], iterations: int):
        """Full search: uniform superposition, `iterations` Grover iterations, measurement."""
        qubits = cudaq.qvector(n)                    # n qubits, all starting in |0>
        h(qubits)                                    # uniform superposition |s> over all 2^n bitstrings
        for _ in range(iterations):                  # changing `iterations` changes P(marked) = sin^2((2k+1)theta)
            grover_iteration(qubits, marked)
        mz(qubits)                                   # measure every qubit in the Z basis

    @cudaq.kernel
    def uniform_superposition(n: int):
        """|s> without measurement: the k = 0 point of the iteration sweep."""
        qubits = cudaq.qvector(n)
        h(qubits)

    @cudaq.kernel
    def continue_iterations(state: cudaq.State, marked: list[int], steps: int):
        """Load a previously computed state and apply `steps` further Grover iterations."""
        qubits = cudaq.qvector(state)                # initialise the register from an existing state vector
        for _ in range(steps):
            grover_iteration(qubits, marked)

    return {"search": grover_search, "uniform": uniform_superposition, "continue": continue_iterations}


kernels_vis = build_grover_kernels()                 # kernel objects used for drawing and statistics only
print("Kernels defined:", ", ".join(kernels_vis))

In [ ]:
# ---------------------------------------------------------------------------------
# Section 7b - Circuit visualization (at most 10 qubits) and full-circuit statistics
# ---------------------------------------------------------------------------------
n_draw = min(int(DRAW_QUBITS), 10, N_QUBITS)         # qubits shown in the drawing (never more than 10)
draw_marked = marked_bits[:n_draw]                    # same marked pattern, truncated to the drawn qubits
print(f"The drawing shows ONE Grover iteration on {n_draw} qubits (marked item {marked_bitstring[:n_draw]}).")
print(f"The executed circuit uses {N_QUBITS} qubits and repeats the iteration {iterations:,} times.\n")
show_circuit(kernels_vis["search"], (n_draw, draw_marked, 1), n_draw, N_QUBITS,
             title=f"Executed circuit: {N_QUBITS} qubits, {iterations} iterations.",
             export=EXPORT_CIRCUIT, formats=EXPORT_FORMATS, out_dir=EXPORT_DIR, basename="grover",
             full_kernel=kernels_vis["search"], full_args=(N_QUBITS, marked_bits, iterations))

# Statistics of the FULL circuit. estimate_resources counts gates without simulating. The circuit
# repeats one block, so counts for 0, 1 and 2 iterations give the exact totals for any k.
stats0 = circuit_statistics(kernels_vis["search"], N_QUBITS, marked_bits, 0)
stats1 = circuit_statistics(kernels_vis["search"], N_QUBITS, marked_bits, 1)
stats2 = circuit_statistics(kernels_vis["search"], N_QUBITS, marked_bits, 2)
if "error" in stats1:
    print(stats1["error"])
else:
    per_iteration = {g: stats1["gate counts by type"].get(g, 0) - stats0["gate counts by type"].get(g, 0)
                     for g in stats1["gate counts by type"]}
    totals = {g: stats0["gate counts by type"].get(g, 0) + iterations * c for g, c in per_iteration.items()}
    depth = stats1["depth"] + (iterations - 1) * (stats2["depth"] - stats1["depth"]) if iterations > 0 else stats0["depth"]
    print(f"\nFULL circuit statistics ({N_QUBITS} qubits, {iterations:,} iterations):")
    print(f"  gate counts by type : {totals}   ('c...cz' = multi-controlled Z with {N_QUBITS - 1} controls)")
    print(f"  total operations    : {sum(totals.values()):,} (including {totals.get('mz', 0)} measurements)")
    print(f"  circuit depth       : {depth:,}")

---
## Section 8 - Execution with live memory monitoring

Running the next cell:
1. initialises the selected backend,
2. builds the quantum circuit (fresh kernels),
3. executes Grover's search with `SHOTS` shots and, optionally, the iteration sweep,
4. monitors host RAM and GPU memory during the whole cell (live line below the cell),
5. prints a short summary. Detailed results follow in Sections 9 and 10.

**How the timings are measured**

| Phase | What happens | How it is measured |
|---|---|---|
| Circuit construction | the Python kernel is translated into CUDA-Q's quantum IR (Quake/MLIR) | CUDA-Q tracer span on the first launch |
| Compilation | MLIR optimisation passes + LLVM just-in-time compilation to machine code | CUDA-Q tracer span on the first launch (cached afterwards) |
| Quantum execution | the simulator (or QPU) applies the gates and measures | tracer span on the first launch + wall-clock time of all later launches |

**How memory is monitored:** a separate lightweight process samples memory every `MEMORY_SAMPLE_INTERVAL_S`
seconds - host RAM used by all processes and this notebook's resident memory (RSS) with `psutil`, and the
memory of every NVIDIA GPU with NVML (device-wide, so it includes other processes on that GPU). Some CUDA-Q
calls (e.g. `cudaq.get_state`) hold Python's interpreter lock, so the *live line* may pause during them,
but the background sampler keeps recording, so the reported peaks are still captured. Peaks shorter than
the sampling interval can be missed. Nothing is estimated or invented: if a monitor is unavailable (e.g. no
NVIDIA driver), the report says so.

In [ ]:
# =====================================================================================
# >>> EXECUTION CELL - Grover's algorithm with live memory monitoring <<<
# =====================================================================================
execution_ok, execution_error = False, None
sweep_iterations, sweep_probabilities, sweep_mode, sweep_time = [], [], "not run", 0.0
monitor = MemoryMonitor(MEMORY_SAMPLE_INTERVAL_S, live=LIVE_MEMORY_DISPLAY)
monitor.start()                                                     # background memory sampling starts now
cell_start = time.perf_counter()
try:
    # Step 1 - initialise the backend (again, so that this cell is self-contained)
    initialize_backend(backend_cfg, seed=SEED)

    # Step 2 - build the circuit: brand-new kernel objects, compiled on their first launch
    kernels = build_grover_kernels()

    # Step 3 - execute the full Grover search; the first launch is traced to separate
    #          construction / compilation / execution time
    counts, phases = profile_first_launch(
        lambda: cudaq.sample(kernels["search"], N_QUBITS, marked_bits, iterations, shots_count=SHOTS))

    # Step 4 - P(marked) as a function of the number of iterations
    sweep_start = time.perf_counter()
    if RUN_ITERATION_SWEEP and uses_state_sweep:
        sweep_mode = "exact amplitudes (state vector evolved iteration by iteration)"
        stride = max(1, math.ceil(sweep_max / SWEEP_MAX_POINTS))    # iterations between recorded points
        state = cudaq.get_state(kernels["uniform"], N_QUBITS)       # k = 0: uniform superposition
        k_done = 0
        sweep_iterations.append(0)
        sweep_probabilities.append(abs(state.amplitude(marked_bits)) ** 2)   # |<x*|psi>|^2
        while k_done + stride <= sweep_max:
            state = cudaq.get_state(kernels["continue"], state, marked_bits, stride)
            k_done += stride
            sweep_iterations.append(k_done)
            sweep_probabilities.append(abs(state.amplitude(marked_bits)) ** 2)
        del state                                                   # free the state vector memory
    elif RUN_ITERATION_SWEEP:
        sweep_mode = f"sampled with {SHOTS} shots per point (this method/backend has no state access)"
        for k in np.unique(np.linspace(0, sweep_max, num=min(SWEEP_MAX_POINTS, 12)).round().astype(int)):
            result_k = cudaq.sample(kernels["search"], N_QUBITS, marked_bits, int(k), shots_count=SHOTS)
            sweep_iterations.append(int(k))
            sweep_probabilities.append(result_k.count(marked_bitstring) / SHOTS)
    sweep_time = time.perf_counter() - sweep_start
    execution_ok = True
except Exception as exc:                                            # report errors instead of crashing later cells
    execution_error = exc
    print(f"\nEXECUTION FAILED: {type(exc).__name__}: {exc}")
    print("Hints: reduce N_QUBITS, choose another BACKEND / SIMULATION_METHOD, or check the GPU driver (README).")
finally:
    memory = monitor.stop()                                         # always stop the sampler
    total_wall = time.perf_counter() - cell_start

if execution_ok:
    phases["total_wall_s"] = total_wall
    measured_marked = counts.count(marked_bitstring)                # shots that returned x*
    p_measured = measured_marked / SHOTS
    most_frequent = counts.most_probable()
    print(f"\nMost frequent bitstring : {most_frequent}  ({'= marked item' if most_frequent == marked_bitstring else 'NOT the marked item'})")
    print(f"Marked-state counts     : {measured_marked} of {SHOTS}  ->  P(marked) = {p_measured:.4f} (theory {p_theory:.4f})")
    print(f"Distinct outcomes       : {len(counts)}")
    print(f"Timing (first launch)   : construction {fmt_seconds(phases['construction_s'])}, "
          f"compilation {fmt_seconds(phases['compilation_s'])}, execution {fmt_seconds(phases['execution_s'])}")
    print(f"Iteration sweep         : {len(sweep_iterations)} points in {sweep_time:.2f} s ({sweep_mode})")
    print(f"Total cell wall time    : {total_wall:.2f} s\n")
print_memory_summary(memory)

---
## Section 9 - Results and visualization

* **Left:** the most frequent measured bitstrings (the marked item highlighted). This is the measured
  probability distribution: counts / `SHOTS`.
* **Right:** probability of the marked item versus the number of Grover iterations - simulated points on top of
  the exact theory curve $\sin^2((2k+1)\theta)$.
* **Bottom:** memory used during the execution cell.

In [ ]:
# ---------------------------------------------------------------------------------
# Section 9 - Measurement table and plots
# ---------------------------------------------------------------------------------
if not execution_ok:
    print("Skipped: the execution cell failed (see the error message above).")
else:
    outcomes = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)   # (bitstring, count), most frequent first
    print(f"{'rank':>4}  {'bitstring':<{max(9, N_QUBITS)}}  {'counts':>7}  {'probability':>11}  marked?")
    for rank, (bits, c) in enumerate(outcomes[:10], start=1):
        print(f"{rank:>4}  {bits:<{max(9, N_QUBITS)}}  {c:>7}  {c / SHOTS:>11.4f}  {'<-- x*' if bits == marked_bitstring else ''}")
    others = SHOTS - measured_marked
    print(f"\nAll other bitstrings together: {others} counts ({others / SHOTS:.4f}); "
          f"theory: {(1 - p_theory):.6f} spread over {search_space - 1:,} bitstrings")
    n_stat = 1.0 / math.sqrt(SHOTS)
    print(f"Statistical resolution of the measured probabilities: about +-{n_stat:.3f} (1/sqrt(SHOTS))")

    fig = plt.figure(figsize=(12, 7.5))
    grid = fig.add_gridspec(2, 2, height_ratios=[1.25, 1.0], hspace=0.45, wspace=0.35)

    # (a) measured distribution: horizontal bars, marked bitstring in blue, everything else muted
    ax_a = fig.add_subplot(grid[0, 0])
    shown = outcomes[:10]                                            # the 10 most frequent outcomes
    n_slots = 10                                                     # fixed number of rows keeps bars thin
    positions = [n_slots - 1 - r for r in range(len(shown))]         # rank 1 at the top
    labels = [b for b, _ in shown]
    values = [c for _, c in shown]
    colors = [BLUE if b == marked_bitstring else INK_MUTED for b in labels]
    ax_a.barh(positions, values, height=0.6, color=colors, edgecolor=SURFACE, linewidth=1.5)
    ax_a.set_yticks(positions, labels, family="DejaVu Sans Mono", fontsize=8)
    ax_a.set_ylim(-0.6, n_slots - 0.4)
    ax_a.set_xlabel(f"counts (of {SHOTS} shots)")
    ax_a.set_title(f"Measured bitstrings: top {len(shown)} of {len(counts)} distinct outcomes", loc="left")
    for pos, v in zip(positions, values):
        ax_a.text(v, pos, f" {v}", va="center", fontsize=8, color=INK_SECONDARY)
    if len(shown) == 1:
        ax_a.text(0.02, 0.35, "every shot returned the same bitstring\n(blue = marked item x*)",
                  transform=ax_a.transAxes, fontsize=9, color=INK_SECONDARY)
    ax_a.set_xlim(0, max(values) * 1.18)
    ax_a.grid(axis="y", visible=False)

    # (b) P(marked) versus iterations: theory curve + simulated points
    ax_b = fig.add_subplot(grid[0, 1])
    k_max_plot = max(sweep_iterations + [iterations, 1])
    k_fine = np.linspace(0, k_max_plot, 600)
    ax_b.plot(k_fine, np.sin((2 * k_fine + 1) * theta) ** 2, color=INK_SECONDARY, linewidth=1.2,
              label=r"theory $\sin^2((2k+1)\theta)$")
    if sweep_iterations:
        ax_b.plot(sweep_iterations, sweep_probabilities, "o", color=BLUE, markeredgecolor=SURFACE,
                  markeredgewidth=1.0, markersize=5, label="CUDA-Q simulation")
    ax_b.plot([iterations], [p_measured], "D", color=ORANGE, markeredgecolor=SURFACE, markersize=8,
              label=f"main run ({SHOTS} shots)")
    ax_b.axvline(k_optimal, color=AXIS_COLOR, linewidth=1)
    ax_b.text(k_optimal, 0.02, f" k_opt = {k_optimal}", ha="left", va="bottom", fontsize=8, color=INK_SECONDARY)
    ax_b.set_ylim(-0.02, 1.1)
    ax_b.set_xlabel("number of Grover iterations k")
    ax_b.set_ylabel("P(marked bitstring)")
    ax_b.set_title("Amplitude amplification and over-rotation", loc="left")
    ax_b.legend(loc="lower center", fontsize=8)

    # (c) memory timeline
    ax_c = fig.add_subplot(grid[1, :])
    monitor.plot(ax=ax_c, title="Memory during the execution cell")
    plt.show()

    if sweep_iterations:
        theory_at_points = np.sin((2 * np.array(sweep_iterations) + 1) * theta) ** 2
        deviation = np.max(np.abs(np.array(sweep_probabilities) - theory_at_points))
        print(f"Sweep ({sweep_mode}): max |simulation - theory| = {deviation:.2e}")

---
## Section 10 - Final benchmark report

The report collects the configuration, the timing breakdown, the measured memory peaks and the comparison
between expected and actual results. It is also saved as `benchmark_report.json` (+ figure) in `EXPORT_DIR`
when `SAVE_REPORT = True`, so runs on different machines and backends can be compared.

In [ ]:
# ---------------------------------------------------------------------------------
# Section 10 - Final benchmark report
# ---------------------------------------------------------------------------------
if not execution_ok:
    print(f"No report: execution failed with {type(execution_error).__name__}: {execution_error}")
else:
    report = build_report("Grover's algorithm (quantum search)", backend_cfg, N_QUBITS, SHOTS, phases, memory,
                          extra_times={"Iteration sweep time (execution)": sweep_time})
    report["Grover iterations (k)"] = f"{iterations:,} (optimal: {k_optimal:,})"
    report["Marked bitstring"] = marked_bitstring
    report["Expected result"] = f"P(marked) = {p_theory:.6f} -> about {expected_counts:.0f} of {SHOTS} shots"
    report["Actual result"] = f"P(marked) = {p_measured:.4f} ({measured_marked}/{SHOTS}); most frequent = {most_frequent}"
    report["Error |P_measured - P_theory|"] = f"{abs(p_measured - p_theory):.4f} (1-sigma shot noise: {count_spread / SHOTS:.4f})"
    if sweep_iterations:
        report["Sweep max deviation from theory"] = f"{deviation:.2e} ({sweep_mode})"
    report["Search succeeded"] = "YES" if most_frequent == marked_bitstring else "NO"

    final_fig, (fa, fb) = plt.subplots(1, 2, figsize=(11, 3.6))
    fa.barh(positions, values, height=0.6, color=colors, edgecolor=SURFACE, linewidth=1.5)
    fa.set_yticks(positions, labels, family="DejaVu Sans Mono", fontsize=8)
    fa.set_ylim(-0.6, n_slots - 0.4)
    fa.set_xlabel("counts")
    fa.set_title(f"Grover N={N_QUBITS}: measured bitstrings", loc="left")
    fa.grid(axis="y", visible=False)
    fb.plot(k_fine, np.sin((2 * k_fine + 1) * theta) ** 2, color=INK_SECONDARY, linewidth=1.2, label="theory")
    if sweep_iterations:
        fb.plot(sweep_iterations, sweep_probabilities, "o", color=BLUE, markeredgecolor=SURFACE, markersize=5, label="simulation")
    fb.set_xlabel("Grover iterations k")
    fb.set_ylabel("P(marked)")
    fb.set_title(f"P(marked) vs k  ({cudaq.get_target().name})", loc="left")
    fb.legend(fontsize=8)
    final_fig.tight_layout()
    show_report(report, save=SAVE_REPORT, out_dir=EXPORT_DIR, name="grover_benchmark_report", figure=final_fig)
    plt.show()

---
### Things to try next
* Increase `N_QUBITS` to 20 / 25 / 30 with `BACKEND = "gpu"` and compare the execution times and memory peaks
  in the saved reports - the state vector doubles with every qubit and $k_{\text{opt}}$ grows by $\sqrt{2}$.
* Set `GPU_PRECISION = "fp32"` to halve the GPU memory and compare the speed.
* Set `GROVER_ITERATIONS` to $k_{\text{opt}}/2$ or $2k_{\text{opt}}$ and check the measured probability against
  the sweep curve.
* `BACKEND = "hardware"` with `HARDWARE_EMULATE = True` compiles the circuit for a real QPU's native gate set and
  runs it in the provider's local emulator (keep N small: multi-controlled gates decompose into many native gates).